# Heterogeneous AFD Disagg Run

Three-step notebook:
1. Initialize config
2. Build the SDK objects from that config
3. Run and print results


In [1]:
import json

cfg = {
    "model_path": "Qwen/Qwen3-30B-A3B",
    "backend_name": "trtllm",
    "database_mode": "HYBRID",
    "prefill_system": "h200_sxm",
    "decode_system": "h200_sxm",
    "prefill_afd_attn_system": "h200_sxm",
    "prefill_afd_ffn_system": "b200_sxm",
    "decode_afd_attn_system": "h200_sxm",
    "decode_afd_ffn_system": "h100_sxm",
    "isl": 4000,
    "osl": 500,
    "runtime_batch_size": 4,
    "prefill_batch_size": 4,
    "decode_batch_size": 100,
    "prefill_num_worker": 2,
    "decode_num_worker": 2,
    "tp": 1,
    "pp": 1,
    "dp": 1,
    "moe_tp": 1,
    "moe_ep": 4,
    "num_attn_gpus": 1,
    "num_ffn_gpus": 4,
    "gpu_layout_strategy": "paired_prefill_decode_per_node"
}

print(json.dumps(cfg, indent=2))


{
  "model_path": "Qwen/Qwen3-30B-A3B",
  "backend_name": "trtllm",
  "database_mode": "HYBRID",
  "prefill_system": "h200_sxm",
  "decode_system": "h200_sxm",
  "prefill_afd_attn_system": "h200_sxm",
  "prefill_afd_ffn_system": "b200_sxm",
  "decode_afd_attn_system": "h200_sxm",
  "decode_afd_ffn_system": "h100_sxm",
  "isl": 4000,
  "osl": 500,
  "runtime_batch_size": 4,
  "prefill_batch_size": 4,
  "decode_batch_size": 100,
  "prefill_num_worker": 2,
  "decode_num_worker": 2,
  "tp": 1,
  "pp": 1,
  "dp": 1,
  "moe_tp": 1,
  "moe_ep": 4,
  "num_attn_gpus": 1,
  "num_ffn_gpus": 4,
  "gpu_layout_strategy": "paired_prefill_decode_per_node"
}


In [2]:
import copy
import json
import logging
import sys
from pathlib import Path

logging.disable(logging.DEBUG)

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "aiconfigurator").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from aiconfigurator.sdk import common, config, perf_database
from aiconfigurator.sdk.backends.factory import get_backend
from aiconfigurator.sdk.inference_session import DisaggInferenceSession

database_mode = common.DatabaseMode[cfg["database_mode"]]

versions = {
    system_name: perf_database.get_latest_database_version(system_name, cfg["backend_name"])
    for system_name in {
        cfg["prefill_system"],
        cfg["decode_system"],
        cfg["prefill_afd_attn_system"],
        cfg["prefill_afd_ffn_system"],
        cfg["decode_afd_attn_system"],
        cfg["decode_afd_ffn_system"],
    }
}

def make_database(system_name: str):
    db = perf_database.get_database(
        system=system_name,
        backend=cfg["backend_name"],
        version=versions[system_name],
    )
    db = copy.deepcopy(db)
    db.set_default_database_mode(database_mode)
    return db

def make_model_config():
    return config.ModelConfig(
        tp_size=cfg["tp"],
        pp_size=cfg["pp"],
        attention_dp_size=cfg["dp"],
        moe_tp_size=cfg["moe_tp"],
        moe_ep_size=cfg["moe_ep"],
        enable_afd=True,
        num_attn_gpus=cfg["num_attn_gpus"],
        num_ffn_gpus=cfg["num_ffn_gpus"],
        gemm_quant_mode=common.GEMMQuantMode.float16,
        moe_quant_mode=common.MoEQuantMode.float16,
        kvcache_quant_mode=common.KVCacheQuantMode.float16,
        fmha_quant_mode=common.FMHAQuantMode.float16,
        comm_quant_mode=common.CommQuantMode.half,
    )

runtime_config = config.RuntimeConfig(
    batch_size=cfg["runtime_batch_size"],
    beam_width=1,
    isl=cfg["isl"],
    osl=cfg["osl"],
    prefix=0,
)

session = DisaggInferenceSession(
    prefill_database=make_database(cfg["prefill_system"]),
    prefill_backend=get_backend(cfg["backend_name"]),
    decode_database=make_database(cfg["decode_system"]),
    decode_backend=get_backend(cfg["backend_name"]),
    prefill_afd_config=config.AfdConfig(
        attn_database=make_database(cfg["prefill_afd_attn_system"]),
        ffn_database=make_database(cfg["prefill_afd_ffn_system"]),
    ),
    decode_afd_config=config.AfdConfig(
        attn_database=make_database(cfg["decode_afd_attn_system"]),
        ffn_database=make_database(cfg["decode_afd_ffn_system"]),
    ),
    gpu_layout_strategy=cfg["gpu_layout_strategy"],
)

run_kwargs = {
    "model_path": cfg["model_path"],
    "runtime_config": runtime_config,
    "prefill_model_config": make_model_config(),
    "prefill_batch_size": cfg["prefill_batch_size"],
    "prefill_num_worker": cfg["prefill_num_worker"],
    "decode_model_config": make_model_config(),
    "decode_batch_size": cfg["decode_batch_size"],
    "decode_num_worker": cfg["decode_num_worker"]
}

print("Session and run arguments are ready.")
print("Resolved database versions:", json.dumps(versions, indent=2))
print("run_kwargs:")
print({
    "model_path": run_kwargs["model_path"],
    "prefill_batch_size": run_kwargs["prefill_batch_size"],
    "decode_batch_size": run_kwargs["decode_batch_size"],
    "prefill_num_worker": run_kwargs["prefill_num_worker"],
    "decode_num_worker": run_kwargs["decode_num_worker"]
})


INFO:aiconfigurator.sdk.perf_database:loading system='h200_sxm', backend='trtllm', version='1.2.0rc5'
INFO:aiconfigurator.sdk.perf_database:loading system='b200_sxm', backend='trtllm', version='1.2.0rc5'
INFO:aiconfigurator.sdk.perf_database:loading system='h100_sxm', backend='trtllm', version='1.2.0rc5'
INFO:aiconfigurator.sdk.inference_session:AstraSim Network Engine enabled for disagg KV cache transfer


Session and run arguments are ready.
Resolved database versions: {
  "b200_sxm": "1.2.0rc5",
  "h100_sxm": "1.2.0rc5",
  "h200_sxm": "1.2.0rc5"
}
run_kwargs:
{'model_path': 'Qwen/Qwen3-30B-A3B', 'prefill_batch_size': 4, 'decode_batch_size': 100, 'prefill_num_worker': 2, 'decode_num_worker': 2}


In [3]:
summary = session.run_disagg(**run_kwargs)

payload = {
    "summary": summary.get_summary_df().iloc[0].to_dict(),
    "network_info": summary.get_network_info()
}

print()
print("ttft_ms            =", payload["summary"]["ttft"])
print("tpot_ms            =", payload["summary"]["tpot"])
print("request_latency_ms =", payload["summary"]["request_latency"])
print("tokens_per_s       =", payload["summary"]["tokens/s"])
print("tokens_per_s_gpu   =", payload["summary"]["tokens/s/gpu"])
print("kv_network_ms      =", payload["summary"]["kv_network_latency_ms"])
print("num_total_gpus     =", payload["summary"]["num_total_gpus"])


INFO:aiconfigurator.sdk.utils:Model architecture: architecture=Qwen3MoeForCausalLM, layers=48, n=32, n_kv=4, d=128, hidden_size=2048, inter_size=6144, vocab=151936, context=40960, topk=8, num_experts=128, moe_inter_size=768, extra_params=None
INFO:aiconfigurator.sdk.inference_session:KV cache transfer size: 1572.86 MB
INFO:aiconfigurator.sdk.inference_session:GPU layout: prefill=[[0], [2]], decode=[[1], [3]]
INFO:aiconfigurator.sdk.astrasim_utils:AstraSim KV cache transfer (tiered): 3.265 ms for 2 transfers, total 1572864000 bytes
INFO:aiconfigurator.sdk.inference_session:Network transfer latency: 3.265 ms



ttft_ms            = 173.961
tpot_ms            = 17.407
request_latency_ms = 8860.054
tokens_per_s       = 10591.96
tokens_per_s_gpu   = 529.598
kv_network_ms      = 3.265208
num_total_gpus     = 20
